In [215]:
import yaml
import numpy as np
from pathlib import Path
from dotmap import DotMap
import matplotlib.pyplot as plt

# Use matplotlib's tex rendering
import matplotlib
matplotlib.rcParams['text.usetex'] = True

#### Load the evaluation configuration

In [216]:
env_type="sc3_staging_15"
basedir = Path("pud/plots/data/" + env_type)

if env_type == "sc2_staging_08":
    config_file = "models/SC2_Staging_08/lag/2024-09-11-19-42-08/bk/config.yaml"
elif env_type == "sc0_staging_20":
    config_file="models/SC0_Staging_20/lag/2024-09-11-19-43-42/bk/config.yaml"
elif env_type == "sc3_staging_05":
    config_file="models/SC3_Staging_05/lag/2024-09-11-19-44-18/bk/config.yaml"
elif env_type == "sc3_staging_11":
    config_file="models/SC3_Staging_11/lag/2024-09-11-15-53-23/bk/config.yaml"
elif env_type == "sc3_staging_15":
    config_file="models/SC3_Staging_15/lag/2024-09-11-19-44-43/bk/config.yaml"

In [217]:
with open(config_file, 'r') as f:
    config = yaml.safe_load(f)
config = DotMap(config)
trained_cost_limit = config.agent_cost_kwargs.cost_limit

In [218]:
problem_type = "easy"
titles = ['Unconstrained', 'Unconstrained\nSearch', 'Constrained', 'Constrained\nSearch (0.1)', 'Constrained\nSearch (0.25)', 'Constrained\nSearch (0.5)', 'Constrained\nSearch (0.75)', 'Constrained\nSearch (1.0)']
colors = ['pink', 'lightblue', 'lightgreen', 'orange', 'gold', 'tomato', 'peachpuff', 'lavender']

#### Extract the metrics from records

In [219]:
def extract_single_agent_metrics(records, search_based=(False, None)):
    success_rate = 0.0
    steps = []
    rewards = []
    cumulative_costs = []
    for idx, record in enumerate(records):
        if len(record) == 0:
            if search_based[0] and search_based[1][idx]["success"]:
                # print("Fallback succeeded")
                success_rate += 1
                steps.append(search_based[1][idx]["steps"])
                rewards.append(search_based[1][idx]["rewards"])
                cumulative_costs.append(search_based[1][idx]["cumulative_costs"])
            # elif search_based[0]:
            #     print("Fallback failed")
            continue
        elif record["success"]:
            success_rate += 1
            steps.append(record["steps"])
            rewards.append(record["rewards"])
            cumulative_costs.append(record["cumulative_costs"])

    metrics = {
        'steps': steps,
        'rewards': rewards,
        'cumulative_costs': cumulative_costs, 
        'success_rate': success_rate / len(records),
    }
    return metrics

In [220]:
def extract_multi_agent_metrics(records, num_agents, search_based=(False, None)):
    success_rate = 0.0
    mean_steps = []
    mean_rewards = []
    mean_cumulative_costs = []
    for idx, record in enumerate(records):
        steps = []
        rewards = []
        successes = []
        cumulative_costs = []
        if len(record[0]) == 0:
            if search_based[0]: 
                for i in range(num_agents):
                    successes.append(search_based[1][idx][i]["success"])
                all_success = all(successes)
                if all_success:
                    success_rate += 1
                    for i in range(num_agents):
                        steps.append(search_based[1][idx][i]["steps"])
                        rewards.append(search_based[1][idx][i]["rewards"])
                        cumulative_costs.append(search_based[1][idx][i]["cumulative_costs"])
                    mean_steps.append(np.mean(steps))
                    mean_rewards.append(np.mean(rewards))
                    mean_cumulative_costs.append(np.max(cumulative_costs))
            continue
        for i in range(num_agents):
            successes.append(record[i]["success"])
        all_success = all(successes)
        if all_success:
            success_rate += 1
            for i in range(num_agents):
                steps.append(record[i]["steps"])
                rewards.append(record[i]["rewards"])
                cumulative_costs.append(record[i]["cumulative_costs"])
        
            mean_steps.append(np.mean(steps))
            mean_rewards.append(np.mean(rewards))
            mean_cumulative_costs.append(np.max(cumulative_costs))

    metrics = {
        'mean_steps': mean_steps,
        'mean_rewards': mean_rewards,
        'mean_cumulative_costs': mean_cumulative_costs,
        'success_rate': success_rate / len(records),
    }

    return metrics

In [221]:
def plot_success_rate(metrics, num_agents):
    unconstrained_sr = metrics[0]['success_rate']
    unconstrained_search_sr = metrics[1]['success_rate']
    constrained_sr = metrics[2]['success_rate']
    constrained_search_srates = [metric["success_rate"] for metric in metrics[3]]

    _, ax = plt.subplots(figsize=(8, 5))

    ax.bar(
        titles,
        [unconstrained_sr, unconstrained_search_sr, constrained_sr, *constrained_search_srates],
        color=colors,
    )
    ax.set_ylim(0, 1)
    ax.set_xlabel('Method', fontsize=14, labelpad=10)
    title_string = rf'\begin{{center}} Success Rate\\[0.01em] $|A| = {num_agents}$ \end{{center}}'
    ax.set_title(title_string, fontsize=16, pad=30)
    ax.set_ylabel('Success Rate', fontsize=14)

    plt.tight_layout()
    plt.show()

In [222]:
def plot_bplots(data, labels, plot_params):
    _, ax = plt.subplots(figsize=(8, 5))

    bplot = ax.boxplot(data, vert=True, notch=False, patch_artist=True, showfliers=False)
    for patch, color in zip(bplot['boxes'], colors):
        patch.set_facecolor(color)  # type: ignore

    ax.yaxis.grid(True)
    ax.set_title(plot_params['title'], fontsize=16, pad=30)
    ax.set_ylabel(plot_params['ylabel'], fontsize=14)
    ax.set_xlabel(plot_params['xlabel'], fontsize=14, labelpad=10)
    ax.set_xticklabels(labels)

    # if 'trained_cost_limit' in plot_params:
    #     ax.axhline(y=plot_params['trained_cost_limit'], color='k', linestyle='--')
    #     ax.annotate(rf'$\bar{{c}} = {plot_params["trained_cost_limit"]}$', xy=(5.05, plot_params['trained_cost_limit'] + 0.1), xytext=(5.05, plot_params['trained_cost_limit'] + 0.5))
        
    plt.tight_layout()
    plt.show()

In [223]:
def plot_steps(metrics, num_agents):
    key = 'mean_steps' if num_agents > 1 else 'steps'
    unconstrained_steps = metrics[0][key]
    unconstrained_search_steps = metrics[1][key]
    constrained_steps = metrics[2][key]
    constrained_search_steps = [metric[key] for metric in metrics[3]]

    title_string = rf'\begin{{center}} Steps\\[0.01em] $|A| = {num_agents}$ \end{{center}}'
    plot_bplots(
        [unconstrained_steps, unconstrained_search_steps, constrained_steps, *constrained_search_steps],
        titles,
        {'title': title_string, 'ylabel': 'Steps', 'xlabel': 'Method'}
    )

In [224]:
def plot_rewards(metrics, num_agents):
    key = 'mean_rewards' if num_agents > 1 else 'rewards'
    unconstrained_rewards = metrics[0][key]
    unconstrained_search_rewards = metrics[1][key]
    constrained_rewards = metrics[2][key]
    constrained_search_rewards = [metric[key] for metric in metrics[3]]

    title_string = rf'\begin{{center}} Rewards\\[0.01em] $|A| = {num_agents}$ \end{{center}}'
    plot_bplots(
        [unconstrained_rewards, unconstrained_search_rewards, constrained_rewards, *constrained_search_rewards],
        titles,
        {'title': title_string, 'ylabel': 'Rewards', 'xlabel': 'Method'}
    )

In [225]:
def plot_cumulative_costs(metrics, num_agents):
    key = 'mean_cumulative_costs' if num_agents > 1 else 'cumulative_costs'
    unconstrained_cc = metrics[0][key]
    unconstrained_search_cc = metrics[1][key]
    constrained_cc = metrics[2][key]
    constrained_search_cc = [metric[key] for metric in metrics[3]]

    # title_string = rf'\begin{{center}} Cumulative Costs\\[0.01em] $|A| = {num_agents}, \bar{{c}} = {trained_cost_limit}$ \end{{center}}'
    title_string = rf'\begin{{center}} Cumulative Costs\\[0.01em] $|A| = {num_agents}$ \end{{center}}'
    plot_bplots(
        [unconstrained_cc, unconstrained_search_cc, constrained_cc, *constrained_search_cc],
        titles,
        {'title': title_string, 'ylabel': 'Cumulative Costs', 'xlabel': 'Method', 'trained_cost_limit': trained_cost_limit}
    )

In [226]:
def plot_everything(metrics, num_agents):
    plot_success_rate(metrics, num_agents)
    plot_steps(metrics, num_agents)
    plot_rewards(metrics, num_agents)
    plot_cumulative_costs(metrics, num_agents)

In [227]:
def means_and_stddevs(metrics, num_agents):
    key = "mean_cumulative_costs" if num_agents > 1 else "cumulative_costs"
    unconstrained_cc = metrics[0][key]
    unconstrained_search_cc = metrics[1][key]
    constrained_cc = metrics[2][key]
    constrained_search_cc = [metric[key] for metric in metrics[3]]
    constrained_search_uc = [metric[key] for metric in metrics[4]]

    unconstrained_cc_mean = np.mean(unconstrained_cc)
    unconstrained_search_cc_mean = np.mean(unconstrained_search_cc)
    constrained_cc_mean = np.mean(constrained_cc)
    constrained_search_cc_means = [np.mean(metric) for metric in constrained_search_cc]
    constrained_search_uc_means = [np.mean(metric) for metric in constrained_search_uc]

    unconstrained_cc_stddev = np.std(unconstrained_cc)
    unconstrained_search_cc_stddev = np.std(unconstrained_search_cc)
    constrained_cc_stddev = np.std(constrained_cc)
    constrained_search_cc_stddevs = [np.std(metric) for metric in constrained_search_cc]
    constrained_search_uc_stddevs = [np.std(metric) for metric in constrained_search_uc]

    edge_cost_factors = [0.1, 0.25, 0.5, 0.75, 1.0]

    print(f"Num Agents: {num_agents}")
    print(f"Unconstrained: {unconstrained_cc_mean:.2f} +/- {unconstrained_cc_stddev:.2f}")
    print(f"Unconstrained Search: {unconstrained_search_cc_mean:.2f} +/- {unconstrained_search_cc_stddev:.2f}")
    print(f"Constrained: {constrained_cc_mean:.2f} +/- {constrained_cc_stddev:.2f}")
    for idx, metric in enumerate(constrained_search_cc_means):
        print(f"Constrained Search ({edge_cost_factors[idx]}): {metric:.2f} +/- {constrained_search_cc_stddevs[idx]:.2f}")
    for idx, metric in enumerate(constrained_search_uc_means):
        print(f"Constrained Search UC ({edge_cost_factors[idx]}): {metric:.2f} +/- {constrained_search_uc_stddevs[idx]:.2f}")

## Single-Agent Comparisons

In [228]:
single_agent_basedir = basedir / "single_agent" / problem_type

unconstrained_records = np.load(single_agent_basedir / "unconstrained_records.npy", allow_pickle=True)
unconstrained_search_records = np.load(single_agent_basedir / "unconstrained_search_records.npy", allow_pickle=True)
constrained_records = np.load(single_agent_basedir / "constrained_records.npy", allow_pickle=True)
constrained_search_factored_records = np.load(single_agent_basedir / "constrained_search_factored_records_uc.npy", allow_pickle=True)
constrained_search_factored_records_cc = np.load(single_agent_basedir / "constrained_search_factored_records.npy", allow_pickle=True)

unconstrained_metrics = extract_single_agent_metrics(unconstrained_records)
unconstrained_search_metrics = extract_single_agent_metrics(unconstrained_search_records, (True, unconstrained_records))
constrained_metrics = extract_single_agent_metrics(constrained_records)
constrained_search_factored_metrics = [extract_single_agent_metrics(csr, (True, constrained_records)) for csr in constrained_search_factored_records]
constrained_search_factored_metrics_cc = [extract_single_agent_metrics(csr, (True, constrained_records)) for csr in constrained_search_factored_records_cc]

# plot_everything([unconstrained_metrics, unconstrained_search_metrics, constrained_metrics, constrained_search_factored_metrics], 1)
means_and_stddevs([unconstrained_metrics, unconstrained_search_metrics, constrained_metrics, constrained_search_factored_metrics_cc, constrained_search_factored_metrics], 1)

Num Agents: 1
Unconstrained: 1.90 +/- 1.76
Unconstrained Search: 2.08 +/- 1.73
Constrained: 1.87 +/- 1.77
Constrained Search (0.1): 1.47 +/- 1.39
Constrained Search (0.25): 1.92 +/- 1.63
Constrained Search (0.5): 1.94 +/- 1.63
Constrained Search (0.75): 1.94 +/- 1.63
Constrained Search UC (0.1): 1.47 +/- 1.39
Constrained Search UC (0.25): 1.79 +/- 1.54
Constrained Search UC (0.5): 1.94 +/- 1.63
Constrained Search UC (0.75): 1.94 +/- 1.63


## Multi-Agent Comparisons

In [229]:
MAX_AGENTS = [5, 10, 20]

multi_agent_basedir = basedir / "multi_agent" / problem_type

for n_agents in MAX_AGENTS:

    print("-" * 50)
    print(f"Number of agents: {n_agents}")
    unconstrained_records = np.load(multi_agent_basedir / f"unconstrained_records_{n_agents}.npy", allow_pickle=True)
    unconstrained_search_records = np.load(multi_agent_basedir / f"unconstrained_search_records_{n_agents}.npy", allow_pickle=True)
    constrained_records = np.load(multi_agent_basedir / f"constrained_records_{n_agents}.npy", allow_pickle=True)
    constrained_search_factored_records = np.load(multi_agent_basedir / f"constrained_search_factored_records_{n_agents}_uc.npy", allow_pickle=True)
    constrained_search_factored_records_cc = np.load(multi_agent_basedir / f"constrained_search_factored_records_{n_agents}.npy", allow_pickle=True)

    unconstrained_metrics = extract_multi_agent_metrics(unconstrained_records, n_agents)
    unconstrained_search_metrics = extract_multi_agent_metrics(unconstrained_search_records, n_agents, (True, unconstrained_records))
    constrained_metrics = extract_multi_agent_metrics(constrained_records, n_agents)
    constrained_search_factored_metrics = [extract_multi_agent_metrics(csr, n_agents, (True, constrained_records)) for csr in constrained_search_factored_records]
    constrained_search_factored_metrics_cc = [extract_multi_agent_metrics(csr, n_agents, (True, constrained_records)) for csr in constrained_search_factored_records_cc]

    # plot_everything([unconstrained_metrics, unconstrained_search_metrics, constrained_metrics, constrained_search_factored_metrics], n_agents)
    means_and_stddevs([unconstrained_metrics, unconstrained_search_metrics, constrained_metrics, constrained_search_factored_metrics_cc, constrained_search_factored_metrics], n_agents)

--------------------------------------------------
Number of agents: 5


FileNotFoundError: [Errno 2] No such file or directory: 'pud/plots/data/sc3_staging_15/multi_agent/easy/constrained_search_factored_records_5_uc.npy'